In [16]:
# Import necessary libraries for efficient implementation
import numpy as np
from collections import Counter, defaultdict
import re

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully!
NumPy version: 1.26.4


In [17]:
# Dataset
documents = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM")
]

print("Dataset loaded successfully!")
print(f"Total documents: {len(documents)}")
for i, (doc, label) in enumerate(documents, 1):
    print(f"{i}. [{label}] {doc}")

Dataset loaded successfully!
Total documents: 11
1. [SPAM] Free money now!!!
2. [HAM] Hi mom, how are you?
3. [SPAM] Lowest price for your meds
4. [HAM] Are we still on for dinner?
5. [SPAM] Win a free iPhone today
6. [HAM] Let's catch up tomorrow at the office
7. [HAM] Meeting at 3 PM tomorrow
8. [SPAM] Get 50% off, limited time!
9. [HAM] Team meeting in the office
10. [SPAM] Click here for prizes!
11. [HAM] Can you send the report?


## Step 1a: Generate Bag of Words (Word Frequency)

In [18]:
def tokenize(text):
    """Convert text to lowercase and split into words using regex"""
    # Use regex to extract alphabetic words AND numbers, converting to lowercase
    tokens = re.findall(r'[a-z0-9]+', text.lower())
    return tokens

# Create bag of words for each class using Counter (efficient counting)
bag_of_words_spam = Counter()
bag_of_words_ham = Counter()
vocabulary = set()  # Using set for O(1) lookup and automatic deduplication

for doc, label in documents:
    tokens = tokenize(doc)
    
    # Add to vocabulary (set automatically handles duplicates)
    vocabulary.update(tokens)
    
    # Update counters based on label
    if label == "SPAM":
        bag_of_words_spam.update(tokens)
    else:  # HAM
        bag_of_words_ham.update(tokens)

print("=== Bag of Words for SPAM ===")
# Counter's most_common() returns sorted by frequency, but we'll sort alphabetically
for word in sorted(bag_of_words_spam.keys()):
    print(f"{word}: {bag_of_words_spam[word]}")

print("\n=== Bag of Words for HAM ===")
for word in sorted(bag_of_words_ham.keys()):
    print(f"{word}: {bag_of_words_ham[word]}")

print(f"\n=== Vocabulary Size: {len(vocabulary)} unique words ===")

=== Bag of Words for SPAM ===
50: 1
a: 1
click: 1
for: 2
free: 2
get: 1
here: 1
iphone: 1
limited: 1
lowest: 1
meds: 1
money: 1
now: 1
off: 1
price: 1
prizes: 1
time: 1
today: 1
win: 1
your: 1

=== Bag of Words for HAM ===
3: 1
are: 2
at: 2
can: 1
catch: 1
dinner: 1
for: 1
hi: 1
how: 1
in: 1
let: 1
meeting: 2
mom: 1
office: 2
on: 1
pm: 1
report: 1
s: 1
send: 1
still: 1
team: 1
the: 3
tomorrow: 2
up: 1
we: 1
you: 2

=== Vocabulary Size: 45 unique words ===


## Step 1b: Calculate Prior Probabilities

In [19]:
# Count documents by class using numpy for efficient counting
labels = np.array([label for _, label in documents])
spam_count = np.sum(labels == "SPAM")
ham_count = np.sum(labels == "HAM")
total_docs = len(documents)

# Calculate priors
prior_spam = spam_count / total_docs
prior_ham = ham_count / total_docs

print(f"Total documents: {total_docs}")
print(f"SPAM documents: {spam_count}")
print(f"HAM documents: {ham_count}")
print(f"\nPrior P(SPAM) = {spam_count}/{total_docs} = {prior_spam:.4f}")
print(f"Prior P(HAM) = {ham_count}/{total_docs} = {prior_ham:.4f}")

Total documents: 11
SPAM documents: 5
HAM documents: 6

Prior P(SPAM) = 5/11 = 0.4545
Prior P(HAM) = 6/11 = 0.5455


## Step 1c: Calculate Likelihood (with Laplace Smoothing)

In [20]:
# Calculate total word counts in each class using sum() on Counter values
total_words_spam = sum(bag_of_words_spam.values())
total_words_ham = sum(bag_of_words_ham.values())

# Laplace smoothing parameter
alpha = 1
vocab_size = len(vocabulary)

# Calculate likelihood for each word in each class using defaultdict
likelihood_spam = {}
likelihood_ham = {}

print("=== Likelihood Calculations (with Laplace Smoothing) ===\n")
print(f"Total words in SPAM: {total_words_spam}")
print(f"Total words in HAM: {total_words_ham}")
print(f"Vocabulary size: {vocab_size}")
print(f"Smoothing parameter (alpha): {alpha}\n")

# Vectorize the likelihood calculations for efficiency
denominator_spam = total_words_spam + alpha * vocab_size
denominator_ham = total_words_ham + alpha * vocab_size

for word in sorted(vocabulary):
    # Get count or 0 if not present
    count_spam = bag_of_words_spam.get(word, 0)
    count_ham = bag_of_words_ham.get(word, 0)
    
    # P(word|class) = (count(word in class) + alpha) / (total words in class + alpha * vocab_size)
    likelihood_spam[word] = (count_spam + alpha) / denominator_spam
    likelihood_ham[word] = (count_ham + alpha) / denominator_ham
    
    print(f"{word:15} | P(w|SPAM)={count_spam + alpha:2}/{denominator_spam:3} = {likelihood_spam[word]:.6f} | "
          f"P(w|HAM)={count_ham + alpha:2}/{denominator_ham:3} = {likelihood_ham[word]:.6f}")

=== Likelihood Calculations (with Laplace Smoothing) ===

Total words in SPAM: 22
Total words in HAM: 34
Vocabulary size: 45
Smoothing parameter (alpha): 1

3               | P(w|SPAM)= 1/ 67 = 0.014925 | P(w|HAM)= 2/ 79 = 0.025316
50              | P(w|SPAM)= 2/ 67 = 0.029851 | P(w|HAM)= 1/ 79 = 0.012658
a               | P(w|SPAM)= 2/ 67 = 0.029851 | P(w|HAM)= 1/ 79 = 0.012658
are             | P(w|SPAM)= 1/ 67 = 0.014925 | P(w|HAM)= 3/ 79 = 0.037975
at              | P(w|SPAM)= 1/ 67 = 0.014925 | P(w|HAM)= 3/ 79 = 0.037975
can             | P(w|SPAM)= 1/ 67 = 0.014925 | P(w|HAM)= 2/ 79 = 0.025316
catch           | P(w|SPAM)= 1/ 67 = 0.014925 | P(w|HAM)= 2/ 79 = 0.025316
click           | P(w|SPAM)= 2/ 67 = 0.029851 | P(w|HAM)= 1/ 79 = 0.012658
dinner          | P(w|SPAM)= 1/ 67 = 0.014925 | P(w|HAM)= 2/ 79 = 0.025316
for             | P(w|SPAM)= 3/ 67 = 0.044776 | P(w|HAM)= 2/ 79 = 0.025316
free            | P(w|SPAM)= 3/ 67 = 0.044776 | P(w|HAM)= 1/ 79 = 0.012658
get             | 

## Step 1d: Classify Test Sentences

In [21]:
def classify_naive_bayes(text):
    """Classify a text using Naive Bayes with numpy for efficient log calculations"""
    tokens = tokenize(text)
    
    print(f"\n{'='*70}")
    print(f"Classifying: '{text}'")
    print(f"Tokens: {tokens}")
    print(f"{'='*70}")
    
    # Start with log probabilities using numpy.log (natural logarithm)
    log_prob_spam = np.log(prior_spam)
    log_prob_ham = np.log(prior_ham)
    
    print(f"\nInitial (Prior):")
    print(f"  log P(SPAM) = log({prior_spam:.4f}) = {log_prob_spam:.6f}")
    print(f"  log P(HAM) = log({prior_ham:.4f}) = {log_prob_ham:.6f}")
    
    print(f"\nLikelihood calculations:")
    for token in tokens:
        if token in vocabulary:
            # Get likelihood for this token
            p_token_spam = likelihood_spam[token]
            p_token_ham = likelihood_ham[token]
            
            log_prob_spam += np.log(p_token_spam)
            log_prob_ham += np.log(p_token_ham)
            
            print(f"  '{token}': P(w|SPAM)={p_token_spam:.6f}, P(w|HAM)={p_token_ham:.6f}")
            print(f"    Running log P(SPAM) = {log_prob_spam:.6f}")
            print(f"    Running log P(HAM) = {log_prob_ham:.6f}")
        else:
            # Unknown word - use smoothing
            p_token_spam = alpha / (total_words_spam + alpha * vocab_size)
            p_token_ham = alpha / (total_words_ham + alpha * vocab_size)
            
            log_prob_spam += np.log(p_token_spam)
            log_prob_ham += np.log(p_token_ham)
            
            print(f"  '{token}' [UNKNOWN]: Using smoothing")
            print(f"    P(w|SPAM)={p_token_spam:.6f}, P(w|HAM)={p_token_ham:.6f}")
    
    print(f"\n{'─'*70}")
    print(f"Final log probabilities:")
    print(f"  log P(SPAM|text) ∝ {log_prob_spam:.6f}")
    print(f"  log P(HAM|text) ∝ {log_prob_ham:.6f}")
    
    # Determine class using numpy comparison
    if log_prob_spam > log_prob_ham:
        prediction = "SPAM"
        confidence = log_prob_spam - log_prob_ham
    else:
        prediction = "HAM"
        confidence = log_prob_ham - log_prob_spam
    
    print(f"\n🎯 PREDICTION: {prediction}")
    print(f"   Confidence (log difference): {confidence:.6f}")
    print(f"{'='*70}")
    
    return prediction

# Test sentences
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

print("\n" + "="*70)
print(" "*20 + "CLASSIFICATION RESULTS")
print("="*70)

results = []
for sentence in test_sentences:
    prediction = classify_naive_bayes(sentence)
    results.append((sentence, prediction))


                    CLASSIFICATION RESULTS

Classifying: 'Limited offer, click here!'
Tokens: ['limited', 'offer', 'click', 'here']

Initial (Prior):
  log P(SPAM) = log(0.4545) = -0.788457
  log P(HAM) = log(0.5455) = -0.606136

Likelihood calculations:
  'limited': P(w|SPAM)=0.029851, P(w|HAM)=0.012658
    Running log P(SPAM) = -4.300003
    Running log P(HAM) = -4.975584
  'offer' [UNKNOWN]: Using smoothing
    P(w|SPAM)=0.014925, P(w|HAM)=0.012658
  'click': P(w|SPAM)=0.029851, P(w|HAM)=0.012658
    Running log P(SPAM) = -12.016241
    Running log P(HAM) = -13.714479
  'here': P(w|SPAM)=0.029851, P(w|HAM)=0.012658
    Running log P(SPAM) = -15.527786
    Running log P(HAM) = -18.083927

──────────────────────────────────────────────────────────────────────
Final log probabilities:
  log P(SPAM|text) ∝ -15.527786
  log P(HAM|text) ∝ -18.083927

🎯 PREDICTION: SPAM
   Confidence (log difference): 2.556141

Classifying: 'Meeting at 2 PM with the manager.'
Tokens: ['meeting', 'at', '2'

## Summary

In [22]:
print("\n" + "="*70)
print(" "*25 + "FINAL SUMMARY")
print("="*70)
print(f"\n📊 Dataset Statistics:")
print(f"   - Total training documents: {len(documents)}")
print(f"   - SPAM documents: {spam_count} ({prior_spam:.1%})")
print(f"   - HAM documents: {ham_count} ({prior_ham:.1%})")
print(f"   - Vocabulary size: {vocab_size} unique words")

print(f"\n📋 Test Results:")
for i, (sentence, prediction) in enumerate(results, 1):
    print(f"   {i}. '{sentence}'")
    print(f"      → Predicted: {prediction}")

print("\n" + "="*70)


                         FINAL SUMMARY

📊 Dataset Statistics:
   - Total training documents: 11
   - SPAM documents: 5 (45.5%)
   - HAM documents: 6 (54.5%)
   - Vocabulary size: 45 unique words

📋 Test Results:
   1. 'Limited offer, click here!'
      → Predicted: SPAM
   2. 'Meeting at 2 PM with the manager.'
      → Predicted: HAM



In [23]:
# Display vocabulary using sorted() for efficient sorting
print("All unique words in vocabulary:")
for i, word in enumerate(sorted(vocabulary), 1):
    print(f"{i}. {word}")
print(f"\nTotal count: {len(vocabulary)}")

All unique words in vocabulary:
1. 3
2. 50
3. a
4. are
5. at
6. can
7. catch
8. click
9. dinner
10. for
11. free
12. get
13. here
14. hi
15. how
16. in
17. iphone
18. let
19. limited
20. lowest
21. meds
22. meeting
23. mom
24. money
25. now
26. off
27. office
28. on
29. pm
30. price
31. prizes
32. report
33. s
34. send
35. still
36. team
37. the
38. time
39. today
40. tomorrow
41. up
42. we
43. win
44. you
45. your

Total count: 45


# Part 2: Using Scikit-Learn
Using the scikit-learn package to train and test a Multinomial Naive Bayes classifier.

## Step 2a: Import Scikit-Learn Libraries

In [32]:
# Import scikit-learn components
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

print("Scikit-learn libraries imported successfully!")

Scikit-learn libraries imported successfully!


## Step 2b: Prepare Training Data

In [33]:
# Prepare training data from the documents
X_train = [doc for doc, _ in documents]
y_train = [label for _, label in documents]

print("Training Data Prepared:")
print(f"- Total samples: {len(X_train)}")
print(f"- Features (X): Text documents")
print(f"- Labels (y): {set(y_train)}")
print(f"\nFirst 3 training examples:")
for i in range(3):
    print(f"  {i+1}. [{y_train[i]}] {X_train[i]}")

Training Data Prepared:
- Total samples: 11
- Features (X): Text documents
- Labels (y): {'HAM', 'SPAM'}

First 3 training examples:
  1. [SPAM] Free money now!!!
  2. [HAM] Hi mom, how are you?
  3. [SPAM] Lowest price for your meds


## Step 2c: Train Multinomial Naive Bayes Model

In [34]:
# Create a pipeline that combines vectorization and classification
# Pipeline steps:
# 1. CountVectorizer: Converts text to word count features (includes numbers with token_pattern)
# 2. MultinomialNB: Trains Naive Bayes classifier with Laplace smoothing (alpha=1.0)

model = Pipeline([
    ('vectorizer', CountVectorizer(
        lowercase=True,           # Convert text to lowercase
        token_pattern=r'[a-z0-9]+'  # Extract words and numbers (same as our manual approach)
    )),
    ('classifier', MultinomialNB(
        alpha=1.0  # Laplace smoothing parameter (same as our manual approach)
    ))
])

# Train the model
model.fit(X_train, y_train)

print("✅ Model trained successfully!")
print("\nModel Details:")
print(f"- Vectorizer: CountVectorizer")
print(f"- Classifier: MultinomialNB")
print(f"- Smoothing (alpha): 1.0")

# Get vocabulary size from the trained vectorizer
vocabulary_sklearn = model.named_steps['vectorizer'].get_feature_names_out()
print(f"- Vocabulary size: {len(vocabulary_sklearn)} unique words")
print(f"\nFirst 10 features: {list(vocabulary_sklearn[:10])}")

✅ Model trained successfully!

Model Details:
- Vectorizer: CountVectorizer
- Classifier: MultinomialNB
- Smoothing (alpha): 1.0
- Vocabulary size: 45 unique words

First 10 features: ['3', '50', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for']


## Step 2d: Classify Test Sentences

In [35]:
# Test sentences
test_sentences_sklearn = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

print("="*70)
print(" "*15 + "SCIKIT-LEARN CLASSIFICATION RESULTS")
print("="*70)

# Classify each test sentence
for i, sentence in enumerate(test_sentences_sklearn, 1):
    # Predict the class
    prediction = model.predict([sentence])[0]
    
    # Get probability estimates for each class
    probas = model.predict_proba([sentence])[0]
    
    # Get class labels
    classes = model.named_steps['classifier'].classes_
    
    print(f"\n{'─'*70}")
    print(f"Test Sentence {i}: '{sentence}'")
    print(f"{'─'*70}")
    print(f"\n🎯 PREDICTION: {prediction}")
    print(f"\nClass Probabilities:")
    for class_label, prob in zip(classes, probas):
        bar_length = int(prob * 50)  # Scale to 50 characters
        bar = '█' * bar_length
        print(f"  {class_label:4} | {bar} {prob:.4f} ({prob*100:.2f}%)")
    
    # Show which class has higher probability
    max_prob_idx = np.argmax(probas)
    confidence = probas[max_prob_idx] - probas[1-max_prob_idx]
    print(f"\nConfidence difference: {confidence:.4f}")

print("\n" + "="*70)

               SCIKIT-LEARN CLASSIFICATION RESULTS

──────────────────────────────────────────────────────────────────────
Test Sentence 1: 'Limited offer, click here!'
──────────────────────────────────────────────────────────────────────

🎯 PREDICTION: SPAM

Class Probabilities:
  HAM  | ████ 0.0838 (8.38%)
  SPAM | █████████████████████████████████████████████ 0.9162 (91.62%)

Confidence difference: 0.8323

──────────────────────────────────────────────────────────────────────
Test Sentence 2: 'Meeting at 2 PM with the manager.'
──────────────────────────────────────────────────────────────────────

🎯 PREDICTION: HAM

Class Probabilities:
  HAM  | ████████████████████████████████████████████████ 0.9781 (97.81%)
  SPAM | █ 0.0219 (2.19%)

Confidence difference: 0.9562



## Step 2e: Compare Manual vs Scikit-Learn Results

In [36]:
# Compare results from manual implementation vs scikit-learn
print("\n" + "="*70)
print(" "*20 + "COMPARISON OF RESULTS")
print("="*70)

print("\n{:<45} {:<12} {:<12}".format("Test Sentence", "Manual", "Scikit-Learn"))
print("-"*70)

for i, sentence in enumerate(test_sentences):
    manual_pred = results[i][1]
    sklearn_pred = model.predict([sentence])[0]
    
    match = "✓ Match" if manual_pred == sklearn_pred else "✗ Different"
    print(f"{sentence:<45} {manual_pred:<12} {sklearn_pred:<12} {match}")

print("\n" + "="*70)
print("\n📊 Summary:")
print(f"   - Manual Implementation: Uses custom bag-of-words and log probability calculations")
print(f"   - Scikit-Learn: Uses optimized CountVectorizer and MultinomialNB")
print(f"   - Both use: Laplace smoothing (alpha=1.0)")
print(f"   - Vocabulary size: {len(vocabulary)} words (manual) vs {len(vocabulary_sklearn)} features (sklearn)")
print("="*70)


                    COMPARISON OF RESULTS

Test Sentence                                 Manual       Scikit-Learn
----------------------------------------------------------------------
Limited offer, click here!                    SPAM         SPAM         ✓ Match
Meeting at 2 PM with the manager.             HAM          HAM          ✓ Match


📊 Summary:
   - Manual Implementation: Uses custom bag-of-words and log probability calculations
   - Scikit-Learn: Uses optimized CountVectorizer and MultinomialNB
   - Both use: Laplace smoothing (alpha=1.0)
   - Vocabulary size: 45 words (manual) vs 45 features (sklearn)
